# Topic 4: Data Quality Audit (ResponsibleAI Guide)
Audits the Week 3 prepared data: `data/processed/prepared_text.jsonl` (chunks from NIST AI 100-1).

Run the cells from top to bottom. Nothing in the repo is changed by this notebook.

In [7]:
# ============================================================
# TOPIC 4 | Cell 1: Load the project data that will be audited
# Reads the files straight from my GitHub repo, so the audit is
# done on the same data that is committed there.
# ============================================================
import json, re, hashlib, urllib.request
from collections import Counter
from pathlib import Path
import pandas as pd

REPO_RAW = "https://raw.githubusercontent.com/emerizzanie/ai-engineering-responsibleai-guide/main/"
AUDIT_DIR = Path("week4_audit")
AUDIT_DIR.mkdir(exist_ok=True)

def fetch(rel_path):
    """Return the bytes of a repo file (local copy if it exists, else GitHub)."""
    local = Path(rel_path)
    if local.is_file():
        return local.read_bytes()
    with urllib.request.urlopen(REPO_RAW + rel_path) as response:
        return response.read()

def sha256(data):
    return hashlib.sha256(data).hexdigest()

# Prepared chunks (Week 3 Part B) and the two manifests
prepared_bytes = fetch("data/processed/prepared_text.jsonl")
records = [json.loads(line) for line in prepared_bytes.decode("utf-8").splitlines() if line.strip()]
manifest = json.loads(fetch("data/week3_manifest.json"))["records"][0]
run_manifest = json.loads(fetch("data/pipeline_v1/documents/pipeline_manifest.json"))
TOTAL_PAGES = run_manifest["inspection"]["pages"]

df = pd.DataFrame(records)
df["length"] = df["text"].str.len()

print("Chunks loaded:", len(df))
print("Fields:", list(records[0].keys()))
print("Source document:", df["source"].unique().tolist(), "| pages in PDF:", TOTAL_PAGES)
print("Prepared file SHA-256 matches the Week 3 manifest:",
      sha256(prepared_bytes) == manifest["prepared_sha256"])


Chunks loaded: 238
Fields: ['record_id', 'text', 'source', 'source_sha256', 'processing', 'page']
Source document: ['nist_ai_100-1_ai_rmf_1.0.pdf'] | pages in PDF: 48
Prepared file SHA-256 matches the Week 3 manifest: True


In [8]:
# ============================================================
# TOPIC 4 | Cell 2: Pick the audit sample (about 18 chunks)
# 12 chunks spread evenly through the document (normal cases)
# + a few unusual ones (very short, very long, cover text,
#   and the chunk with the most hyphenated words).
# The choice is fixed by rules, so it is the same on every run.
# ============================================================
HYPHEN_WORD = re.compile(r"\b[A-Za-z]+-[A-Za-z]+\b")
df["hyphen_words"] = df["text"].apply(lambda t: len(HYPHEN_WORD.findall(t)))

picked = {}  # index -> reason

# Normal cases: every 20th chunk, starting from chunk 10
for i in range(9, len(df), 20):
    picked[i] = "evenly spaced (normal case)"

# Unusual cases
for i in df["length"].nsmallest(2).index:
    picked[i] = "one of the 2 shortest chunks"
picked[df["length"].idxmax()] = "longest chunk"
picked[0] = "first chunk (cover page)"
picked[1] = "second chunk (cover text again?)"
picked[len(df) - 1] = "last chunk (back page)"
picked[df["hyphen_words"].idxmax()] = "most hyphenated words"

sample = df.loc[sorted(picked)].copy()
sample["why_picked"] = [picked[i] for i in sample.index]
sample["text_start"] = sample["text"].str.slice(0, 70)

pd.set_option("display.max_colwidth", 80, "display.width", 200)
print("Sample size:", len(sample), "of", len(df), "chunks")
print(sample[["record_id", "page", "length", "why_picked", "text_start"]].to_string(index=False))

sample.drop(columns=["text_start"]).to_csv(AUDIT_DIR / "week4_audit_sample.csv", index=False)
print("\nSaved:", AUDIT_DIR / "week4_audit_sample.csv")


Sample size: 19 of 238 chunks
                    record_id  page  length                       why_picked                                                             text_start
nist_ai_100-1_ai_rmf_1.0-0001     1      76         first chunk (cover page) NIST AI 100-1 Artificial Intelligence Risk Management Framework (AI RM
nist_ai_100-1_ai_rmf_1.0-0002     2     376 second chunk (cover text again?) NIST AI 100-1 Artificial Intelligence Risk Management Framework (AI RM
nist_ai_100-1_ai_rmf_1.0-0010     5     581      evenly spaced (normal case) List of Figures Fig. 1 Examples of potential harms related to AI syste
nist_ai_100-1_ai_rmf_1.0-0015     6     488            most hyphenated words risks can emerge in a variety of ways and can be characterized as long
nist_ai_100-1_ai_rmf_1.0-0018     6     689                    longest chunk These risks make AI a uniquely challenging technology to deploy and ut
nist_ai_100-1_ai_rmf_1.0-0030     9     437      evenly spaced (normal case) Part 

In [9]:
# ============================================================
# TOPIC 4 | Cell 3: Evidence for the five quality dimensions
# Each check is printed for the SAMPLE and for ALL chunks,
# so I can keep the two apart in the report.
# Timeliness cannot be checked by code (see the note at the end).
# ============================================================
FOOTER = re.compile(r"\bPage \d+\b")
REQUIRED = ["record_id", "text", "source", "source_sha256", "processing", "page"]

def nospace(s):
    return re.sub(r"\s+", "", s)

def report(name, data):
    print(f"\n===== {name}: {len(data)} chunks =====")

    # 1. COMPLETENESS: is required information missing?
    missing = {f: int((data[f].isna() | (data[f].astype(str).str.strip() == "")).sum()) for f in REQUIRED}
    tiny = data[data["length"] < 50]
    print("COMPLETENESS")
    print("  missing values per field:", missing)
    print("  chunks under 50 characters:", len(tiny), tiny["record_id"].str[-4:].tolist())

    # 2. CONSISTENCY: do related records agree?
    print("CONSISTENCY")
    print("  different source names:", data["source"].nunique(),
          "| different source fingerprints:", data["source_sha256"].nunique(),
          "| different processing labels:", data["processing"].nunique())
    print("  source fingerprint equals the manifest:",
          bool((data["source_sha256"] == manifest["source_sha256"]).all()))

    # 3. ACCURACY: does the text faithfully represent the source?
    footer_chunks = int(data["text"].str.contains(FOOTER).sum())
    print("ACCURACY")
    print("  chunks that still contain a 'Page N' footer:", footer_chunks)
    print("  hyphenated words:", int(data["hyphen_words"].sum()),
          "| likely broken words (estimate):", int(data["broken_words"].sum()))

    # 5. UNIQUENESS: are unnecessary duplicates present?
    print("UNIQUENESS")
    print("  exact duplicate texts:", int(data["text"].duplicated().sum()))
    print("  duplicate record IDs:", int(data["record_id"].duplicated().sum()))

# A hyphenated word is counted as "likely broken" when the same word
# without the hyphen also appears in the document (e.g. sys-tem / system).
# This is an estimate, not an exact count.
all_words = Counter(re.findall(r"[a-z]+", " ".join(df["text"]).lower()))
def broken_words(text):
    return [w for w in HYPHEN_WORD.findall(text) if all_words[w.lower().replace("-", "")] > 0]
df["broken_words"] = df["text"].apply(lambda t: len(broken_words(t)))
sample["broken_words"] = df.loc[sample.index, "broken_words"]

report("SAMPLE", sample)
report("ALL CHUNKS", df)

# Extra checks that only make sense on the whole file
print("\n===== WHOLE FILE ONLY =====")
pages_present = set(df["page"])
print("Pages with no chunk:", [p for p in range(1, TOTAL_PAGES + 1) if p not in pages_present])
print("Page numbers never go backwards:", bool(df["page"].is_monotonic_increasing))

# Near-duplicates: a chunk whose whole text also sits inside another chunk
texts = df["text"].tolist()
inside = [(df["record_id"][a][-4:], df["record_id"][b][-4:])
          for a in range(len(texts)) for b in range(len(texts))
          if a != b and len(texts[a]) > 30 and texts[a] in texts[b]]
print("Chunk text repeated inside another chunk (small, big):", inside)

# Consistency between the Part B file and the Part C file
part_c = [json.loads(l) for l in fetch("data/pipeline_v1/documents/prepared_text.jsonl").decode("utf-8").splitlines() if l.strip()]
print("Part B and Part C have the same text and pages:",
      [r["text"] for r in part_c] == texts and [r["page"] for r in part_c] == df["page"].tolist())
print("Part B 'processing' label:", repr(df["processing"].iloc[0]))
print("Part C 'processing' label:", repr(part_c[0]["processing"]))

print("\nExamples of likely broken words:",
      [w for w, _ in Counter(w for t in df["text"] for w in broken_words(t)).most_common(8)])
print("The 2 shortest chunks:", df.nsmallest(2, "length")[["record_id", "page", "text"]].values.tolist())

print("\nTIMELINESS: not checked by code. I need to look up by hand whether")
print("NIST AI 100-1 (January 2023) is still the current version.")



===== SAMPLE: 19 chunks =====
COMPLETENESS
  missing values per field: {'record_id': 0, 'text': 0, 'source': 0, 'source_sha256': 0, 'processing': 0, 'page': 0}
  chunks under 50 characters: 2 ['0052', '0066']
CONSISTENCY
  different source names: 1 | different source fingerprints: 1 | different processing labels: 1
  source fingerprint equals the manifest: True
ACCURACY
  chunks that still contain a 'Page N' footer: 4
  hyphenated words: 28 | likely broken words (estimate): 14
UNIQUENESS
  exact duplicate texts: 0
  duplicate record IDs: 0

===== ALL CHUNKS: 238 chunks =====
COMPLETENESS
  missing values per field: {'record_id': 0, 'text': 0, 'source': 0, 'source_sha256': 0, 'processing': 0, 'page': 0}
  chunks under 50 characters: 3 ['0052', '0066', '0237']
CONSISTENCY
  different source names: 1 | different source fingerprints: 1 | different processing labels: 1
  source fingerprint equals the manifest: True
ACCURACY
  chunks that still contain a 'Page N' footer: 42
  hyphenated wor

In [10]:
# ============================================================
# TOPIC 4 | Cell 4: Accuracy check against the source PDF
# For each sample chunk: is its text really on the PDF page
# that the chunk says it came from?
# Spaces and line breaks are ignored in the comparison.
# ============================================================
!pip -q install pypdf
import io
from pypdf import PdfReader

# Read the PDF from the repo. If it is not in the repo, use my Google Drive copy
# (mount Drive first and change the folder path if needed).
PDF_BACKUP = Path("/content/drive/MyDrive/AI Engineering/ai_engineering_data/nist_ai_100-1_ai_rmf_1.0.pdf")
try:
    pdf_bytes = fetch("data/raw/nist_ai_100-1_ai_rmf_1.0.pdf")
except Exception:
    pdf_bytes = PDF_BACKUP.read_bytes()
print("PDF SHA-256 matches the Week 3 manifest:", sha256(pdf_bytes) == manifest["source_sha256"])

reader = PdfReader(io.BytesIO(pdf_bytes))
page_text = {i + 1: (p.extract_text() or "") for i, p in enumerate(reader.pages)}

def found_on_page(row):
    return nospace(row["text"]) in nospace(page_text[row["page"]])

sample["found_on_stated_page"] = sample.apply(found_on_page, axis=1)
df["found_on_stated_page"] = df.apply(found_on_page, axis=1)

print("SAMPLE: chunks found on their stated page:", int(sample["found_on_stated_page"].sum()), "of", len(sample))
print("ALL:    chunks found on their stated page:", int(df["found_on_stated_page"].sum()), "of", len(df))
print("Not found:", df.loc[~df["found_on_stated_page"], "record_id"].tolist())
print("\nNote: this compares with the text that pypdf extracts. It does not check")
print("figures or tables, so I also open a few PDF pages and compare by eye.")



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


PDF SHA-256 matches the Week 3 manifest: True
SAMPLE: chunks found on their stated page: 19 of 19
ALL:    chunks found on their stated page: 238 of 238
Not found: []

Note: this compares with the text that pypdf extracts. It does not check
figures or tables, so I also open a few PDF pages and compare by eye.


In [11]:
# ============================================================
# TOPIC 4 | Cell 5: Two validation rules (Activity 3)
# Rule 1: every chunk has a valid source reference.
# Rule 2: every chunk has real content (not only a page footer).
# A failed chunk is listed for review. Nothing is deleted.
# ============================================================
import numbers

def rule_1_valid_source(row):
    """Source name present, fingerprint equals the manifest, page is a real page."""
    return (
        isinstance(row["source"], str) and row["source"].strip() != ""
        and row["source_sha256"] == manifest["source_sha256"]
        and isinstance(row["page"], numbers.Integral) and not isinstance(row["page"], bool)
        and 1 <= row["page"] <= TOTAL_PAGES
    )

MIN_CONTENT = 50
def rule_2_real_content(row):
    """At least 50 characters are left after the 'Page N' footer is removed."""
    content = FOOTER.sub("", row["text"]).strip()
    return len(content) >= MIN_CONTENT

rows = df.to_dict("records")
df["rule_1_pass"] = [rule_1_valid_source(r) for r in rows]
df["rule_2_pass"] = [rule_2_real_content(r) for r in rows]

for rule, title in [("rule_1_pass", "RULE 1: valid source reference"),
                    ("rule_2_pass", "RULE 2: real content (50+ characters without the footer)")]:
    in_sample = df.loc[sample.index, rule]
    print(title)
    print(f"  Sample: {int(in_sample.sum())} pass, {int((~in_sample).sum())} fail (of {len(in_sample)})")
    print(f"  All:    {int(df[rule].sum())} pass, {int((~df[rule]).sum())} fail (of {len(df)})")
    failed = df[~df[rule]]
    for _, r in failed.iterrows():
        print(f"    FAIL {r['record_id']} | page {r['page']} | {r['text'][:60]!r}")
    print()

failed_any = df[~(df["rule_1_pass"] & df["rule_2_pass"])]
failed_any[["record_id", "page", "length", "text", "rule_1_pass", "rule_2_pass"]].to_csv(
    AUDIT_DIR / "week4_review_list.csv", index=False)
print("Chunks sent for human review:", len(failed_any), "->", AUDIT_DIR / "week4_review_list.csv")


RULE 1: valid source reference
  Sample: 19 pass, 0 fail (of 19)
  All:    238 pass, 0 fail (of 238)

RULE 2: real content (50+ characters without the footer)
  Sample: 17 pass, 2 fail (of 19)
  All:    234 pass, 4 fail (of 238)
    FAIL nist_ai_100-1_ai_rmf_1.0-0052 | page 12 | 'Page 7'
    FAIL nist_ai_100-1_ai_rmf_1.0-0066 | page 14 | 'Page 9'
    FAIL nist_ai_100-1_ai_rmf_1.0-0151 | page 31 | 'Categories Subcategories Continued on next page Page 26'
    FAIL nist_ai_100-1_ai_rmf_1.0-0237 | page 47 | 'in particular. Page 42'

Chunks sent for human review: 4 -> week4_audit\week4_review_list.csv


In [12]:
# ============================================================
# TOPIC 4 | Cell 6: Coverage check (Activity 4)
# Part 1: how much of each source document is prepared.
# Part 2: how many chunks mention the topics a student may ask about.
# This is a simple keyword count. It shows if a topic is present,
# not how well it is explained.
# ============================================================

# Part 1: page counts come from my Week 3 inventory (docs/week3_data_inventory.md)
SOURCE_PAGES = {
    "S01 NIST AI 100-1": 48,
    "S02 NIST AI 600-1": 64,
    "S03 Malaysia AIGE": 140,
    "S04 ASEAN Guide": 87,
    "S05 UNESCO GenAI": 48,
}
chunks_per_source = {name: 0 for name in SOURCE_PAGES}
chunks_per_source["S01 NIST AI 100-1"] = len(df)   # the only prepared document

total_pages = sum(SOURCE_PAGES.values())
print("PART 1: What is prepared")
for name, pages in SOURCE_PAGES.items():
    print(f"  {name:20s} pages: {pages:4d} | prepared chunks: {chunks_per_source[name]}")
print(f"  Pages prepared: {SOURCE_PAGES['S01 NIST AI 100-1']} of {total_pages} "
      f"({SOURCE_PAGES['S01 NIST AI 100-1'] / total_pages:.0%})")

# Part 2: topics grouped by user need
TOPICS = {
    "Need 1: general AI risks": ["privacy", "bias", "fair", "transparen", "safe", "secur", "accountab"],
    "Need 2: Malaysian / ASEAN guidance": ["Malaysia", "ASEAN"],
    "Extra: generative AI and students": ["generative", "student", "education"],
}

print("\nPART 2: Chunks that mention each topic (of", len(df), "chunks)")
for need, words in TOPICS.items():
    print(" ", need)
    for word in words:
        hit = df["text"].str.contains(word, case=False, regex=False)
        print(f"    {word:12s} chunks: {int(hit.sum()):3d} | pages: {df.loc[hit, 'page'].nunique():2d}")

# Part 3: are some key words split by a line-end hyphen more than others?
# "broken" = the word shows up with a hyphen inside it (e.g. trustworthi-ness).
all_text = " ".join(df["text"]).lower()
hyphen_tokens = Counter(re.findall(r"\b[a-z]+(?:-[a-z]+)+\b", all_text))

print("\nPART 3: Key words written correctly vs broken by a hyphen")
for word in ["privacy", "bias", "transparency", "accountability", "trustworthiness"]:
    correct = len(re.findall(r"\b" + word + r"\b", all_text))
    broken = sum(n for token, n in hyphen_tokens.items() if token.replace("-", "") == word)
    print(f"    {word:16s} correct: {correct:3d} | broken: {broken}")


PART 1: What is prepared
  S01 NIST AI 100-1    pages:   48 | prepared chunks: 238
  S02 NIST AI 600-1    pages:   64 | prepared chunks: 0
  S03 Malaysia AIGE    pages:  140 | prepared chunks: 0
  S04 ASEAN Guide      pages:   87 | prepared chunks: 0
  S05 UNESCO GenAI     pages:   48 | prepared chunks: 0
  Pages prepared: 48 of 387 (12%)

PART 2: Chunks that mention each topic (of 238 chunks)
  Need 1: general AI risks
    privacy      chunks:  18 | pages: 12
    bias         chunks:  21 | pages: 12
    fair         chunks:  11 | pages:  9
    transparen   chunks:  26 | pages: 16
    safe         chunks:  18 | pages: 12
    secur        chunks:  20 | pages: 12
    accountab    chunks:  17 | pages: 13
  Need 2: Malaysian / ASEAN guidance
    Malaysia     chunks:   0 | pages:  0
    ASEAN        chunks:   0 | pages:  0
  Extra: generative AI and students
    generative   chunks:   2 | pages:  2
    student      chunks:   0 | pages:  0
    education    chunks:   0 | pages:  0

PART 3: Ke

In [13]:
# ============================================================
# TOPIC 4 | Cell 7: Responsible data checks (Activity 5)
# Part 1: simple search for contact details inside the chunks.
# Part 2: why my Week 3 inventory and my manifest show different character counts.
# Part 3: print the rights statement that is written inside each source PDF.
# Run after Cell 4 (it uses page_text from that cell).
# ============================================================
EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
PHONE = re.compile(r"\+?\d[\d\s().-]{8,}\d")

print("PART 1: Contact details inside the", len(records), "chunks (simple search)")
for name, pattern in [("email addresses", EMAIL), ("phone-like numbers", PHONE)]:
    found = [(r["record_id"][-4:], m) for r in records for m in pattern.findall(r["text"])]
    print(f"  {name}: {len(found)}", found)
print("  Note: this search only finds emails and phone numbers. It does not find names.")

print("\nPART 2: Character count of S01")
chars_pages = sum(len(t) for t in page_text.values())
chars_joined = len("\n".join(page_text.values()))
print("  48 pages added up:", chars_pages, "(the number in my Week 3 inventory is 106,446)")
print("  48 pages joined with a line break:", chars_joined,
      "| pipeline manifest says:", run_manifest["inspection"]["characters"])
print("  difference:", chars_joined - chars_pages, "= the line breaks between the pages")

# Part 3: the page where each PDF prints its rights statement (PDF page number)
RIGHTS_PAGE = {
    "nist_ai_100-1_ai_rmf_1.0.pdf": 2,
    "nist_ai_600-1_genai_profile.pdf": 2,
    "malaysia_aige_2024.pdf": 2,
    "asean_ai_governance_ethics_2024.pdf": 87,
    "unesco_genai_guidance_2023.pdf": 2,
}
RIGHTS_WORDS = re.compile(r"free of charge|all rights reserved|protected by copyright|open access", re.I)

def read_pdf(name):
    """Use my local copy in data/raw if it exists, else the copy in my GitHub repo."""
    for folder in [Path("data/raw"), Path("../data/raw")]:
        if (folder / name).is_file():
            return (folder / name).read_bytes()
    return fetch("data/raw/" + name)

print("\nPART 3: Rights statement inside each PDF")
for name, page_no in RIGHTS_PAGE.items():
    text = PdfReader(io.BytesIO(read_pdf(name))).pages[page_no - 1].extract_text() or ""
    text = re.sub(r"\s+", " ", text)
    match = RIGHTS_WORDS.search(text)
    start = max(0, match.start() - 40) if match else 0
    print(f"  {name} (PDF page {page_no}):")
    print("    ", text[start:start + 300] if match else "no rights wording found on this page")


PART 1: Contact details inside the 238 chunks (simple search)
  email addresses: 1 [('0006', 'AIframework@nist.gov')]
  phone-like numbers: 0 []
  Note: this search only finds emails and phone numbers. It does not find names.

PART 2: Character count of S01
  48 pages added up: 106446 (the number in my Week 3 inventory is 106,446)
  48 pages joined with a line break: 106493 | pipeline manifest says: 106493
  difference: 47 = the line breaks between the pages

PART 3: Rights statement inside each PDF
  nist_ai_100-1_ai_rmf_1.0.pdf (PDF page 2):
      RMF 1.0) This publication is available free of charge from: https://doi.org/10.6028/NIST.AI.100-1 January 2023 U.S. Department of Commerce Gina M. Raimondo, Secretary National Institute of Standards and Technology Laurie E. Locascio, NIST Director and Under Secretary of Commerce for Standards and T
  nist_ai_600-1_genai_profile.pdf (PDF page 2):
     e Profile This publication is available free of charge from: https://doi.org/10.6028/NIST.A